# 🏋️ 用户行为轨迹 体验诊断 — 自我练习版

> 参考答案在 `37_业务健康度诊断实战_电商评论文本挖掘.ipynb`

---

## Step 0: 环境准备 & 数据导入（已完成）

In [7]:
# !kaggle datasets download nicapotato/womens-ecommerce-clothing-reviews --path ./data --unzip -f 'Womens Clothing E-Commerce Reviews.csv'

In [8]:
# !pip install nltk scikit-learn -q

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

import platform
if platform.system() == 'Darwin':
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS']
else:
    plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', palette='muted')
print('✅ 环境配置完成！')

✅ 环境配置完成！


In [10]:
# 加载数据（自动兼容文件名空格问题）
DATA_DIR = './data'
candidates = ['Womens Clothing E-Commerce Reviews.csv', 'Womens%20Clothing%20E-Commerce%20Reviews.csv']
data_path = None
for name in candidates:
    path = os.path.join(DATA_DIR, name)
    if os.path.exists(path):
        data_path = path
        break
if data_path is None:
    raise FileNotFoundError(f'数据文件未找到，请检查 {DATA_DIR} 目录')

df = pd.read_csv(data_path, index_col=0)
print(f'📊 数据集大小: {df.shape}')
print(f'📊 列名: {list(df.columns)}')
df.head()

📊 数据集大小: (23486, 10)
📊 列名: ['Clothing ID', 'Age', 'Title', 'Review Text', 'Rating', 'Recommended IND', 'Positive Feedback Count', 'Division Name', 'Department Name', 'Class Name']


,Clothing ID,Age,Title,Review Text,Rating,Recommended IND,Positive Feedback Count,Division Name,Department Name,Class Name
0,767,33,NaN,Absolutely wonderful - silky and sexy and comf...,4,1,0,Initmates,Intimate,Intimates
1,1080,34,NaN,Love this dress! it's sooo pretty. i happene...,5,1,4,General,Dresses,Dresses
2,1077,60,Some major design flaws,I had such high hopes for this dress and reall...,3,0,0,General,Dresses,Dresses
3,1049,50,My favorite buy!,"I love, love, love this jumpsuit. it's fun, fl...",5,1,0,General Petite,Bottoms,Pants
4,847,47,Flattering shirt,This shirt is very flattering to all due to th...,5,1,6,General,Tops,Blouses


---
## Step 1: 数据探索 & 理解业务语境
**任务**: 了解评分分布、品类分布，定义差评阈值

In [30]:
# TODO: 1.1 数据概览 — 查看缺失值和数据类型
df.columns = df.columns.str.lower()
df.isnull().sum() #title,review text,division name,department name,class name     
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 23486 entries, 0 to 23485
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   clothing id              23486 non-null  int64
 1   age                      23486 non-null  int64
 2   title                    19676 non-null  str  
 3   review text              22641 non-null  str  
 4   rating                   23486 non-null  int64
 5   recommended ind          23486 non-null  int64
 6   positive feedback count  23486 non-null  int64
 7   division name            23472 non-null  str  
 8   department name          23472 non-null  str  
 9   class name               23472 non-null  str  
dtypes: int64(5), str(5)
memory usage: 1.8 MB


In [31]:
df.describe(include = 'all')

,clothing id,age,title,review text,rating,recommended ind,positive feedback count,division name,department name,class name
count,23486.000000,23486.000000,19676,22641,23486.000000,23486.000000,23486.000000,23472,23472,23472
unique,NaN,NaN,13993,22634,NaN,NaN,NaN,3,6,20
top,NaN,NaN,Love it!,Perfect fit and i've gotten so many compliment...,NaN,NaN,NaN,General,Tops,Dresses
freq,NaN,NaN,136,3,NaN,NaN,NaN,13850,10468,6319
mean,918.118709,43.198544,NaN,NaN,4.196032,0.822362,2.535936,NaN,NaN,NaN
std,203.298980,12.279544,NaN,NaN,1.110031,0.382216,5.702202,NaN,NaN,NaN
min,0.000000,18.000000,NaN,NaN,1.000000,0.000000,0.000000,NaN,NaN,NaN
25%,861.000000,34.000000,NaN,NaN,4.000000,1.000000,0.000000,NaN,NaN,NaN
50%,936.000000,41.000000,NaN,NaN,5.000000,1.000000,1.000000,NaN,NaN,NaN
75%,1078.000000,52.000000,NaN,NaN,5.000000,1.000000,3.000000,NaN,NaN,NaN


In [ ]:
# TODO: 1.2 评分分布可视化 + 定义 NEGATIVE_THRESHOLD
# pd.cut(df['rating'],bins=5).value_counts() 
rating_dis = df['rating'].value_counts(normalize=True).reset_index()
rating_dis


,rating,proportion
0,5,0.559099
1,4,0.216171
2,3,0.122243
3,2,0.066635
4,1,0.035851


In [55]:
rating_dis['proportion'].cumsum()
# 结论：后10%（评分2及以下判断为差评
NEGATIVE_THRESHOLD = 2



In [57]:
df['department name'].unique()

<StringArray>
['Intimate', 'Dresses', 'Bottoms', 'Tops', 'Jackets', 'Trend', nan]
Length: 7, dtype: str

In [ ]:
# TODO: 1.3 品类分布可视化 (Division / Department / Class)





division name   department name  class name    
General         Bottoms          Casual bottoms    0.000144
                                 Jeans             0.056968
                                 Pants             0.117256
                                 Shorts            0.140144
                                 Skirts            0.183538
                Dresses          Dresses           0.452852
                Jackets          Jackets           0.481516
                                 Outerwear         0.499422
                Tops             Blouses           0.643321
                                 Fine gauge        0.692780
                                 Knits             0.928087
                                 Sweaters          0.993069
                Trend            Trend             1.000000
General Petite  Bottoms          Jeans             1.044335
                                 Pants             1.112438
                                 Skirts            1

---
## Step 2: 文本预处理
**任务**: 构建 `preprocess_text()` 函数，包含分词→去停用词→词形还原

In [76]:
# TODO: 2.1 构建文本预处理管道
# 提示: 使用 WordNetLemmatizer, stopwords, word_tokenize
# 别忘了添加业务自定义停用词
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# 添加业务自定义停用词（电商场景高频但无意义的词）
CUSTOM_STOPWORDS = {
    'dress', 'top', 'wear', 'would', 'like', 'one', 'also',
    'really', 'look', 'get', 'got', 'go', 'make', 'made',
    'much', 'even', 'still', 'try', 'tried', 'could',
}
stop_words = stop_words.union(CUSTOM_STOPWORDS)

def preprocess_text(text:str) -> str:
    if pd.isna(text):
        return '' 
    
    text = text.lower()

    tokens = word_tokenize(text)

    tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
        if token.isalpha() and token not in stop_words and len(token) > 2 
    ]

    return ' '.join(tokens)




In [78]:
# TODO: 2.2 批量预处理 + 添加 is_negative 标签
df_text = df.dropna(subset=['review text']).copy()
print(f'有文本的评论: {len(df_text):,} / {len(df):,}')

df_text['clean_text'] = df_text['review text'].apply(preprocess_text)

df_text = df_text[df_text['clean_text'].str.strip() != ''].copy()
print(f'处理后有效文本: {len(df_text):,}')

df_text['is_negative'] = (df_text['rating'] <= NEGATIVE_THRESHOLD).astype(int)
print(f'\n差评占比: {df_text["is_negative"].mean():.2%}')

df_text[['review text','clean_text','rating','is_negative']].head()



有文本的评论: 22,641 / 23,486
处理后有效文本: 22,641

差评占比: 10.47%


,review text,clean_text,rating,is_negative
0,Absolutely wonderful - silky and sexy and comf...,absolutely wonderful silky sexy comfortable,4,0
1,Love this dress! it's sooo pretty. i happene...,love sooo pretty happened find store glad neve...,5,0
2,I had such high hopes for this dress and reall...,high hope wanted work initially ordered petite...,3,0
3,"I love, love, love this jumpsuit. it's fun, fl...",love love love jumpsuit fun flirty fabulous ev...,5,0
4,This shirt is very flattering to all due to th...,shirt flattering due adjustable front tie perf...,5,0


---
## Step 3: TF-IDF 高频词提取
**任务**: 分别对差评和好评做 TF-IDF，对比 Top 15 关键词

In [79]:
# TODO: 3.1 TF-IDF 提取差评 vs 好评关键词
negative_texts = df_text[df_text['is_negative'] == 1]['clean_text']
positive_texts = df_text[df_text['is_negative'] == 0]['clean_text']

TFIDF_MAX_FEATURES = 500
TFIDF_MIN_DF = 5
TFIDF_NGRAM_RANGE = (1,2)

tfidf = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    min_df=TFIDF_MIN_DF,
    ngram_range=TFIDF_NGRAM_RANGE
)

tfidf_neg = tfidf.fit_transform(negative_texts)
neg_feature_names = tfidf.get_feature_names_out()
neg_scores = tfidf_neg.mean(axis=0).A1
neg_top = pd.Series(neg_scores, index=neg_feature_names).sort_values(ascending=False)

tfidf_pos = tfidf.fit_transform(positive_texts)
pos_feature_names = tfidf.get_feature_names_out()
pos_scores = tfidf_pos.mean(axis=0).A1
pos_top = pd.Series(pos_scores, index=pos_feature_names).sort_values(ascending=False)

print('\n🔴 差评 Top 15 关键词:')
print(neg_top.head(15))
print('\n🟢 好评 Top 15 关键词:')
print(pos_top.head(15))


🔴 差评 Top 15 关键词:
fabric      0.040851
size        0.039448
fit         0.039384
back        0.035080
color       0.032739
shirt       0.030934
small       0.030219
material    0.029338
love        0.028996
ordered     0.028693
look        0.027648
looked      0.026213
way         0.025837
quality     0.024025
cute        0.023100
dtype: float64

🟢 好评 Top 15 关键词:
love           0.051015
fit            0.049102
size           0.046785
great          0.041244
color          0.039972
fabric         0.029627
small          0.028890
perfect        0.028759
little         0.027149
comfortable    0.026898
flattering     0.026809
soft           0.025774
cute           0.025089
ordered        0.024606
beautiful      0.024311
dtype: float64


In [17]:
# TODO: 3.2 可视化对比


---
## Step 4: LDA 主题聚类
**任务**: 对差评做 LDA，发现 5 个隐藏主题，然后手动命名

In [81]:
# TODO: 4.1 LDA 训练（注意用 CountVectorizer 不是 TfidfVectorizer）
N_TOPICS = 5
RANDOM_STATE = 42

count_vec = CountVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    min_df=TFIDF_MIN_DF,
    ngram_range=TFIDF_NGRAM_RANGE
)
neg_count_matrix = count_vec.fit_transform(negative_texts)

lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=RANDOM_STATE,
    max_iter=20,
    learning_method='online'
)
lda.fit(neg_count_matrix)

feature_names = count_vec.get_feature_names_out()
WORDS_PER_TOPIC = 10

print('='*60)
print(f'🔍 LDA 差评主题分析 ({N_TOPICS} 个主题)')
print('='*60)

topic_labels = {}  # 存储主题标签，后续手动命名
for topic_idx, topic in enumerate(lda.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-WORDS_PER_TOPIC-1:-1]]
    print(f'\n📌 Topic {topic_idx}: {" | ".join(top_words)}')
    # 根据关键词手动命名主题（面试时这就是你的"业务判断力"）
    topic_labels[topic_idx] = f'Topic_{topic_idx}'




🔍 LDA 差评主题分析 (5 个主题)

📌 Topic 0: material | quality | fabric | fit | run | color | small | way | size | jean

📌 Topic 1: back | shirt | going | love | look | sweater | going back | looked | wanted | online

📌 Topic 2: time | first | loved | pant | bought | washed | wore | wash | shirt | retailer

📌 Topic 3: size | fit | small | ordered | large | arm | medium | way | fabric | chest

📌 Topic 4: fabric | cheap | color | look | material | looked | back | felt | blouse | photo


In [19]:
# TODO: 4.2 根据关键词手动命名主题


In [20]:
# TODO: 4.3 为每条差评分配主题 + 可视化主题分布


---
## Step 5: 交叉分析
**任务**: 做 主题 × Division 和 主题 × Department 的交叉表 + 热力图

In [21]:
# TODO: 5.1 交叉表: 主题 × Division


In [22]:
# TODO: 5.2 热力图: 主题 × Department


In [23]:
# TODO: 5.3 各 Division 差评率对比


---
## Step 6: YoY 模拟分析
**任务**: 用年龄段模拟 YoY，看不同用户群的痛点结构差异

In [24]:
# TODO: 6.1 模拟 YoY 分析


---
## Step 7: 优先级排序
**任务**: 构建优先级评分模型（差评占比 × 影响范围 × 不推荐率）

In [25]:
# TODO: 7.1 优先级评分


In [26]:
# TODO: 7.2 气泡图可视化


---
## 📋 结论与建议
**TODO**: 写出你的分析结论和行动建议